In [1]:
from training import count_parameters, TrainConfig, train_waitk_student, TranslationDataset
import mlflow
import torch
import json
from evaluation import SimulMTEvaluator, MTQualityScorer, WaitKHybridMamba2Adapter
from model_classes import WaitKHybridMamba2MT, SimulHybridMamba2Config

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("simulmt_waitk_hybrid_distillation")

from transformers import AutoTokenizer

teacher_name = "./models/nllb_teacher"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)

In [3]:
hybrid_config = SimulHybridMamba2Config(
    vocab_size = len(tokenizer),
    d_model = 512,
    num_encoder_layers = 6,
    num_decoder_layers = 6,
    d_state = 128,
    d_conv = 4,
    expand = 2,
    headdim = 64,
    ngroups = 1,
    nhead = 8,
    dim_feedforward = 2048,
    dropout = 0.1,
    max_source_len = 64,
    max_target_len = 64,
    pad_token_id = tokenizer.pad_token_id,
    eos_token_id = tokenizer.eos_token_id,
)

In [4]:
hybrid_mamba2 = WaitKHybridMamba2MT(hybrid_config)

In [5]:
count_parameters(hybrid_mamba2)

Total parameters:     301,988,416
Trainable parameters: 301,988,416
Frozen parameters:    0


{'total': 301988416, 'trainable': 301988416, 'frozen': 0}

In [6]:
train_cfg = TrainConfig(
    epochs=10,
    short_epochs=False,
    batches_per_epoch=2000,
    batch_size=192,
    gradient_accumulation_steps=8,

    wait_k=[3, 5, 7, 9, 11],

    use_kl_loss=False,
    use_dataset_ce_loss=True,

    kl_weight=1.0,
    dataset_ce_weight=1.0,

    lr=2e-5,
    use_amp=True,
)

In [7]:
dataset = TranslationDataset("./data/train_dataset.hdf5")

In [9]:
train_waitk_student(
    student=hybrid_mamba2,
    train_dataset=dataset,
    model_cfg=hybrid_config,
    train_cfg=train_cfg,
    device="cuda",
    checkpoint_name_prefix="hybrid",
    resume_from_checkpoint="./checkpoints/hybrid_step_12000.pt"
)

epoch 8/10:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 9/10:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 10/10:   0%|          | 0/14504 [00:00<?, ?it/s]

🏃 View run waitk-student at: http://localhost:5000/#/experiments/3/runs/b76a5c8a73b64b7d9899cd1ce49d9144
🧪 View experiment at: http://localhost:5000/#/experiments/3


KeyboardInterrupt: 

In [ ]:
eval_dataset = TranslationDataset(
    "./data/val_dataset.hdf5",
    lazy=False,
)

adapter = WaitKHybridMamba2Adapter(
    model=hybrid_mamba2,
    tokenizer=tokenizer,
    name="hybrid_wait_k",
    device="cuda",
    use_amp=True,
)

evaluator = SimulMTEvaluator(
    tokenizer=tokenizer,
    quality_scorer=MTQualityScorer(),
)

for k in [5, 7, 9, 11]:
    result = evaluator.evaluate(
        model=adapter,
        dataset=eval_dataset,
        wait_k=k,
        batch_size=1024,
        max_new_tokens=63
    )
    
    print(result.metrics)
    
    with open(f"./metrics/hybrid_k{k}.json", "w") as file:
        json.dump(result.metrics, file)

Validating hybrid_wait_k, wait_k=5:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 34.47525047299003, 'chrF++': 56.16746652209361, 'TER': 58.26195547840047, 'AP': 0.7304746368559976, 'AL': 6.3215980513560845, 'DAL': 7.0995424374257965, 'LAAL': 6.178942626368124, 'ATD_text': 4.450144690416921, 'total_time_sec': 379.57191362600133, 'ms_per_sentence': 1.22676032974371, 'target_tokens_per_sec': 23360.77744871529, 'source_tokens_per_sec': 20505.368602348648, 'first_token_latency_sec': None, 'peak_gpu_memory_mb': 4870.98583984375, 'generation_total_time_sec': 329.99743358399064, 'generation_ms_per_sentence': 1.0665377123686717, 'generation_target_tokens_per_sec': 26870.193818471485}


Validating hybrid_wait_k, wait_k=7:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 35.05895832416267, 'chrF++': 56.674609591437516, 'TER': 57.31736448765344, 'AP': 0.7862514333791565, 'AL': 8.138808263484663, 'DAL': 8.925775570438663, 'LAAL': 7.437746455647556, 'ATD_text': 5.65545762744165, 'total_time_sec': 380.2887407820017, 'ms_per_sentence': 1.229077084716078, 'target_tokens_per_sec': 23255.526792126788, 'source_tokens_per_sec': 20466.716905672758, 'first_token_latency_sec': None, 'peak_gpu_memory_mb': 4870.98583984375, 'generation_total_time_sec': 329.66451556398533, 'generation_ms_per_sentence': 1.0654617354448315, 'generation_target_tokens_per_sec': 26826.711952513688}


Validating hybrid_wait_k, wait_k=9:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 35.21261044488232, 'chrF++': 56.81123059374358, 'TER': 57.02862931791092, 'AP': 0.8310896291815405, 'AL': 9.89782528535633, 'DAL': 10.67209052889656, 'LAAL': 8.531122227283733, 'ATD_text': 6.720109272773502, 'total_time_sec': 381.256707960998, 'ms_per_sentence': 1.2322055135936074, 'target_tokens_per_sec': 23171.922789890297, 'source_tokens_per_sec': 20414.75425213034, 'first_token_latency_sec': None, 'peak_gpu_memory_mb': 4870.98583984375, 'generation_total_time_sec': 329.97886167500474, 'generation_ms_per_sentence': 1.066477688746339, 'generation_target_tokens_per_sec': 26772.778580892937}


Validating hybrid_wait_k, wait_k=11:   0%|          | 0/303 [00:00<?, ?it/s]